## 📌 本版測試內容（E7 notebook 版本 2026-11-27-r2，#766）

- **目標**：Colab（Python 3.13、L4）上 **一次 Run All**（`SMOKE_MODE` True／False 都一樣），中間不用重啟、不用手動改格。
- **套件版本重定（D384 更新）**：`torch==2.7.1` + `torchvision==0.22.1`（兩者都有 cp313 wheel，Python 3.13 有 Dynamo，laya 的 ModernBERT `@torch.compile` 可載入）、`transformers==4.53.3`、`huggingface_hub==0.33.4`、`accelerate==1.8.1`、`safetensors==0.5.3`。**numpy／Pillow／matplotlib 不安裝、不降版**，只檢查不低於下限。
- **CELL 2 版本一致檢查**：安裝後比對「硬碟上的版本」與「記憶體已載入的版本」，不一致就停並印出「重啟 runtime 後再 Run All」。
- **上傳包改為單一 zip**：Drive 上的 `e7-colab-pack.zip`（旁邊有 `.zip.sha256`）先解到 `/content/e7-colab-pack`，再在本機磁碟驗證 MANIFEST（不再直接讀 Drive 的 4252 個小檔）。
- **續跑**：CELL 6 每完成一組 (party, mode) 就寫 checkpoint 到 `OUTPUT_DIR/checkpoints/`；重跑時 fingerprint（commit、範圍、請求數、套件版本）相同的組直接跳過，已全完成的模型不重新載入。
- **打包**：最後一格（CELL 9）把輸出、checkpoint、`RUN_NOTES.json`（版本、偏離、SHA-256）打成 `OUTPUT_DIR/e7-run-<日期>.zip`。
- **出處**：Qwen 的 `hf_revision` 固定為 commit `66285546d2b821cf421d4f5eb2576359d3770cd3`；權重 SHA-256 算不出來時記 `UNKNOWN`＋原因（#768）。
- **版本**：notebook 日期 `2026-11-27-r2`；程式 commit `a3c7e90d24ea149ada3eee75be2d02618d975f1d`。CELL 1B 印版本橫幅，checkout 不符就停。
- **資料**：E3/E5 舊 16 段（`e3_16_segments`），531 個請求（以 `PACK_INFO` 為準）。
- **會跑的決策器與模式**：
  - `qwen_vl`：`image_text` 與 `text_only`
  - `laya_text`、`laya_typed`：`text_only`
- **跳過**：`diffusion_vlm`（LLaDA-V，自訂程式未審，記為 `SKIPPED_UNAUDITED`）。
- **加速對照**：每組跑加速（CUDA）與不加速（CPU，前 20 個決策點）。
- **不呼叫付費 API、不上傳 GT**；答案表只在 Cell 7 讀。
- **雲端硬碟位置**：輸入 `我的雲端硬碟/amidst/e7-colab-pack.zip`（＋`.sha256`），輸出 `我的雲端硬碟/amidst/e7-outputs/`。
- **建議 GPU**：L4 或 A100。

---

# E7 Local / Colab Decider Evaluation (`#727` / `#731`, D384–D387 / D392)

**One-click Run All** notebook for evaluating local/open-weights deciders on Colab Pro:
- **One-click condition:** Out of the box, Run All completes end-to-end (including Cell 7 scoring and Cell 8 export) for all audited models (`Qwen/Qwen2.5-VL-3B-Instruct`). `laya` is audited (HF commit `246177e5…`, native `RLAgent` path, only 3 audited .py files with SHA-256 check, no `pip install laya`, no `trust_remote_code`) and runs, as two parties: `laya_text` (root English checkpoint, main) and `laya_typed` (`typed-decisions/` fine-tuned checkpoint; README reports 0.766 vs 0.362 for base English on typed-decisions; same repo/commit, zero API cost). Unaudited custom-code models (`LLaDA-V`) are recorded as `SKIPPED_UNAUDITED` and skipped cleanly without aborting. To include LLaDA-V, audit its HF custom `.py` files and set `custom_code_audited = True` + pinned `audited_commit` in Cell 1.
- **Deciders:** small `Qwen-VL` (`Qwen/Qwen2.5-VL-3B-Instruct`), `laya_text` / `laya_typed` text-only (`convaiinnovations/laya`, root vs `typed-decisions/` checkpoint, native ModernBERT decision-head path via audited `rl_agent_api.RLAgent`, not a CausalLM), and 1 vision-capable diffusion LM (`GSAI-ML/LLaDA-V`, with mandatory HF custom `.py` audit gate).
- **Dataset scopes:** supports `e3_16_segments` (old 16 segments, exploratory), `e6_12_segments` (E6 new 12 segments, formal comparison additional group), or `combined`.
- **Acceleration comparison:** runs **both** accelerated (CUDA bf16 if `torch.cuda.is_bf16_supported()` else fp16; actual dtype recorded in Cell 3) and unaccelerated (CPU, capped to `CPU_SAMPLE_CAP` decision points when slow) for each model, with explicit warmup outside the timed loop (`load_ms` recorded separately).
- **GT Isolation (D385 / D392):** Label-only answer table (without `gt_regime`) is read **ONLY** in Cell 7 (Scoring Cell) and hard-fails if missing or if any forbidden GT field (`gt_visible`, `gt_boxes`, `gt_regime`, etc.) is present. Decider code (`e7_decider_eval.py`) never receives or reads answer tables.
- **Identity status:** `UNCONFIRMED`. No paid API calls.

In [ ]:
# === CELL 1: CENTRALIZED CONFIGURATION (Edit only here if needed) ===
from pathlib import Path

# Repository & pinned git commit for reproducible Colab execution
# NOTE (official #732 runs): PINNED_GIT_COMMIT and every model's hf_revision MUST be replaced
# with exact 40-char commit SHAs before executing a formal #732 evaluation run.
GIT_REPO_URL = "https://github.com/Polalalabear/amidst.git"
PINNED_GIT_COMMIT = "a3c7e90d24ea149ada3eee75be2d02618d975f1d"  # exact 40-char SHA; Cell 1B stops if the checkout differs
NOTEBOOK_VERSION_DATE = "2026-11-27-r2"

# One-click Run All condition:
# - Audited models (e.g. Qwen2.5-VL-3B-Instruct) run through Cell 7 & Cell 8 automatically.
# - Unaudited custom-code models (LLaDA-V) are recorded as SKIPPED_UNAUDITED (no crash).
# - laya is audited (#727, commit 246177e5). To include LLaDA-V, audit its HF custom .py files and set custom_code_audited=True + audited_commit.

# Upload pack: ONE zip on Drive (+ sidecar <zip>.sha256). CELL 4 unzips it to local disk and verifies MANIFEST there.
PACK_ZIP = Path("/content/drive/MyDrive/amidst/e7-colab-pack.zip")
PACK_ROOT = Path("/content/e7-colab-pack")  # local extraction target (fast disk); CELL 4 may point it at a nested folder
OUTPUT_DIR = Path("/content/drive/MyDrive/amidst/e7-outputs")
REPO_DIR = Path("/content/amidst")

# Set SMOKE_MODE = True for offline/CI dry-run with deterministic synthetic backend (no GPU / HF download)
SMOKE_MODE = False

# Dataset scope: "e3_16_segments" | "e6_12_segments" | "combined"
DATASET_SCOPE = "e3_16_segments"

# Input modes to run per model (image_text = primary, text_only = ablation)
INPUT_MODES = ["image_text", "text_only"]

# Acceleration comparison toggles
RUN_ACCELERATED = True
RUN_UNACCELERATED = True
ACCELERATED_DTYPE = "cuda_bf16"
CPU_SAMPLE_CAP = 20  # Fixed small-sample cap for slow CPU unaccelerated comparison

# Model configurations (hf_revision="main" or exact commit SHA; runtime resolution records exact commit & weight SHA-256s)
MODEL_CONFIGS = [
    {
        "party_id": "qwen_vl",
        "adapter": "QwenVLDecider",
        "hf_repo_id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "hf_revision": "66285546d2b821cf421d4f5eb2576359d3770cd3",
        "custom_code_audited": True,
        "audited_commit": None,
        "modes": ["image_text", "text_only"],
    },
    {
        "party_id": "laya_text",  # root English checkpoint (main laya party)
        "adapter": "LayaTextDecider",
        "hf_repo_id": "convaiinnovations/laya",
        "hf_revision": "246177e5cb70334c9831c0730864a93c836aac8f",
        "checkpoint_subdir": "",
        "custom_code_audited": True,  # audited (#727): only rl_common.py, rl_agent_api.py, email_utils.py, SHA-256 checked at load
        "audited_commit": "246177e5cb70334c9831c0730864a93c836aac8f",
        "modes": ["text_only"],
    },
    {
        "party_id": "laya_typed",  # same repo/commit; fine-tuned typed-decisions checkpoint (README: 0.766 vs 0.362 base English)
        "adapter": "LayaTextDecider",
        "hf_repo_id": "convaiinnovations/laya",
        "hf_revision": "246177e5cb70334c9831c0730864a93c836aac8f",
        "custom_code_audited": True,
        "audited_commit": "246177e5cb70334c9831c0730864a93c836aac8f",
        "checkpoint_subdir": "typed-decisions",
        "modes": ["text_only"],
    },
    {
        "party_id": "diffusion_vlm",
        "adapter": "DiffusionVLMDecider",
        "hf_repo_id": "GSAI-ML/LLaDA-V",
        "hf_revision": "main",
        "custom_code_audited": False,  # MUST audit HF custom .py files and set True + audited_commit before running
        "audited_commit": None,
        "modes": ["image_text", "text_only"],
    },
]

# Pinned dependency versions (D384, re-pinned for Colab Python 3.13, #766).
# - torch 2.7.1 / torchvision 0.22.1: matching pair, both have cp313 manylinux wheels (PyPI); torch >= 2.6 has Dynamo on 3.13.
# - transformers 4.53.3 needs huggingface_hub >=0.30,<1.0 -> 0.33.4.
# - numpy / Pillow / matplotlib are NOT installed or downgraded (Colab's loaded copies stay valid); only minimums are checked.
# - The notebook adds src/ to sys.path without installing the amidst wheel, so pyproject.toml's (higher) numpy/pillow minimums do not apply.
PINNED_DEPS = [
    "torch==2.7.1",
    "torchvision==0.22.1",
    "transformers==4.53.3",
    "huggingface_hub==0.33.4",
    "safetensors==0.5.3",
    "accelerate==1.8.1",
]
MIN_RUNTIME_VERSIONS = {"numpy": "2.0.0", "Pillow": "10.0.0", "matplotlib": "3.8.0"}

# Per-tick laya run (#767): set RUN_PARTIES = ["laya_text", "laya_typed"], RUN_UNACCELERATED = False, SCORE_ON_COLAB = False
# and upload the requests-only pack (no answer table, no frames). Scoring then happens on the Mac (D385). Defaults = the full E7 run.
RUN_PARTIES = None  # None = every audited party; or a list of party_id
SCORE_ON_COLAB = True  # False: skip CELL 7 (no answer table on Colab); CELL 8 writes the raw comparisons instead

# Free-text deviations from the plan (e.g. "ran with L4 instead of A100"); copied into RUN_NOTES.json by CELL 9.
RUN_DEVIATIONS: list[str] = []

In [ ]:
# === CELL 1B: MOUNT GOOGLE DRIVE (COLAB) & CLONE / CHECKOUT PINNED REPO ===
import subprocess

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Mounted Google Drive at /content/drive")
except ImportError:
    print("Not running in Google Colab; skipping drive.mount('/content/drive') (local fallback).")

if (Path.cwd() / "src" / "amidst").is_dir():
    REPO_DIR = Path.cwd()
    print(f"Already inside repo checkout: {REPO_DIR}; skipping git clone.")
elif not (REPO_DIR / ".git").is_dir():
    print(f"Cloning {GIT_REPO_URL} -> {REPO_DIR} ...")
    subprocess.check_call(["git", "clone", GIT_REPO_URL, str(REPO_DIR)])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", PINNED_GIT_COMMIT])
else:
    print(f"Repo already present at {REPO_DIR}; checking out {PINNED_GIT_COMMIT} ...")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", PINNED_GIT_COMMIT])

# --- Version banner & pin check (Colab clone / checkout paths only; a local in-repo run just reports) ---
_head = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
_parties = [c["party_id"] for c in MODEL_CONFIGS if c.get("custom_code_audited") and (RUN_PARTIES is None or c["party_id"] in RUN_PARTIES)]
_skipped = [c["party_id"] for c in MODEL_CONFIGS if not c.get("custom_code_audited") and (RUN_PARTIES is None or c["party_id"] in RUN_PARTIES)]
print("=" * 72)
print(f"E7 NOTEBOOK {NOTEBOOK_VERSION_DATE} | repo commit {_head}")
print(f"RUNNING: {', '.join(_parties)} | SKIPPED (unaudited): {', '.join(_skipped) or 'none'}")
print("=" * 72)
if not (Path.cwd() / "src" / "amidst").is_dir() and _head != PINNED_GIT_COMMIT:
    raise RuntimeError(f"STOP: checked-out commit {_head} != PINNED_GIT_COMMIT {PINNED_GIT_COMMIT}")


In [ ]:
# === CELL 2: INSTALL PINNED DEPENDENCIES + MEMORY/DISK VERSION CONSISTENCY CHECK ===
import subprocess, sys

if (REPO_DIR / "src").is_dir() and str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from amidst.evaluation.e7_colab_runtime import assert_runtime_consistent, check_min_versions

if not SMOKE_MODE:
    print("Installing pinned dependencies:", PINNED_DEPS)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINNED_DEPS])
    # Stops with "重啟 runtime 後再 Run All" if torch/transformers/... were already imported at another version.
    RUNTIME_VERSIONS = assert_runtime_consistent(PINNED_DEPS)
    RUNTIME_VERSIONS["runtime_minimums"] = check_min_versions(MIN_RUNTIME_VERSIONS)
    print("Runtime versions consistent (memory == disk == pin):", RUNTIME_VERSIONS["disk"])
else:
    RUNTIME_VERSIONS = {"pins": {}, "disk": {}, "loaded": {}, "note": "SMOKE_MODE: pip install and version check skipped"}
    print("SMOKE_MODE=True; skipping pip install of PINNED_DEPS.")

In [ ]:
# === CELL 3: HARDWARE & ENVIRONMENT PROVENANCE ===
import json
from amidst.evaluation.e7_decider_eval import collect_hardware_provenance

HARDWARE_PROVENANCE = collect_hardware_provenance(REPO_DIR if REPO_DIR.is_dir() else None)
actual_dtype = HARDWARE_PROVENANCE.get("actual_accelerated_dtype", "fp32")
ACCELERATED_DTYPE = f"cuda_{actual_dtype}" if HARDWARE_PROVENANCE.get("gpu_name", "NONE") != "NONE" and actual_dtype != "fp32" else f"cpu_{actual_dtype}"
print(f"CUDA bf16 supported: {HARDWARE_PROVENANCE.get('cuda_bf16_supported')} | actual_accelerated_dtype: {actual_dtype} | label: {ACCELERATED_DTYPE}")
print(json.dumps(HARDWARE_PROVENANCE, indent=2, ensure_ascii=False))


In [ ]:
# === CELL 4: UNZIP UPLOAD PACK TO LOCAL DISK, THEN MANIFEST SHA-256 VERIFICATION (STOPS ON MISMATCH) ===
from amidst.evaluation.e7_colab_runtime import extract_pack_zip
from amidst.evaluation.e7_decider_eval import verify_manifest_sha256

PACK_ZIP_SHA256 = None
if PACK_ZIP.is_file():
    _ex = extract_pack_zip(PACK_ZIP, PACK_ROOT)  # checks the zip against <zip>.sha256, refuses path traversal
    PACK_ROOT = _ex["pack_root"]
    PACK_ZIP_SHA256 = _ex["zip_sha256"]
    print(f"Unzipped {PACK_ZIP.name} (sha256 {PACK_ZIP_SHA256}; checked against: {_ex['sha256_checked_against']}) -> {PACK_ROOT}")
elif (PACK_ROOT / "MANIFEST.sha256").is_file():
    print(f"No zip at {PACK_ZIP}; using already-extracted pack at {PACK_ROOT}.")
else:
    raise FileNotFoundError(f"PACK_ZIP_MISSING: {PACK_ZIP} (upload e7-colab-pack.zip and its .sha256 to Drive)")

MANIFEST_VERIFICATION = verify_manifest_sha256(PACK_ROOT)
print("MANIFEST verified:", MANIFEST_VERIFICATION)

In [ ]:
# === CELL 5: LOAD INFERENCE REQUESTS (DECIDER SIDE — NO ANSWER TABLE READ HERE) ===
import json
from amidst.evaluation.e7_decider_eval import DeciderRequest, filter_requests_by_scope

def load_requests_jsonl(pack_root: Path, scope: str) -> list[DeciderRequest]:
    files = []
    if scope in ("e3_16_segments", "combined"):
        files.append(pack_root / "requests" / "e3_16_segments.jsonl")
    if scope in ("e6_12_segments", "combined"):
        files.append(pack_root / "requests" / "e6_12_segments.jsonl")
    out = []
    for fpath in files:
        if not fpath.is_file():
            continue
        for line in fpath.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            d = json.loads(line)
            out.append(
                DeciderRequest(
                    request_id=d["request_id"],
                    case_id=d["case_id"],
                    dataset_group=d["dataset_group"],
                    tick=int(d["tick"]),
                    variable_text=d["variable_text"],
                    candidates=tuple(d.get("candidates") or []),
                    camera_ids=tuple(d.get("camera_ids") or []),
                    image_paths={k: str(pack_root / v) for k, v in (d.get("image_paths") or {}).items()},
                )
            )
    return filter_requests_by_scope(out, scope)

REQUESTS = load_requests_jsonl(PACK_ROOT, DATASET_SCOPE)
print(f"Loaded {len(REQUESTS)} inference requests for scope={DATASET_SCOPE}")


In [ ]:
# === CELL 6: RUN DECIDERS (RESUMABLE; ACCELERATED vs UNACCELERATED; REAL HF BACKENDS OR SMOKE; NO GT ACCESS) ===
from amidst.evaluation.e7_colab_runtime import load_group_checkpoint, run_fingerprint, save_group_checkpoint
from amidst.evaluation.e7_decider_eval import (
    CustomCodeAuditError,
    DiffusionVLMDecider,
    LayaTextDecider,
    QwenVLDecider,
    build_diffusion_vlm_inference_fn,
    build_laya_native_inference_fn,
    build_qwen_vl_inference_fn,
    build_smoke_inference_fn,
    resolve_hf_provenance,
    run_acceleration_comparison,
)

ADAPTER_MAP = {
    "LayaTextDecider": LayaTextDecider,
    "QwenVLDecider": QwenVLDecider,
    "DiffusionVLMDecider": DiffusionVLMDecider,
}

FINGERPRINT = run_fingerprint(
    git_commit=HARDWARE_PROVENANCE.get("git_commit", "UNKNOWN"),
    dataset_scope=DATASET_SCOPE,
    n_requests=len(REQUESTS),
    smoke_mode=SMOKE_MODE,
    cpu_sample_cap=CPU_SAMPLE_CAP,
    pinned_deps=PINNED_DEPS,
    run_unaccelerated=RUN_UNACCELERATED,
    pack_manifest_sha256=MANIFEST_VERIFICATION.get("manifest_sha256"),
)


def build_decider(cfg):
    """Create the decider (loads nothing heavy until warmup); raises CustomCodeAuditError if unaudited."""
    cls = ADAPTER_MAP[cfg["adapter"]]
    kwargs = {"hf_repo_id": cfg["hf_repo_id"], "hf_revision": cfg["hf_revision"]}
    if cfg["adapter"] in ("LayaTextDecider", "DiffusionVLMDecider"):
        kwargs["custom_code_audited"] = cfg["custom_code_audited"]
        kwargs["audited_commit"] = cfg["audited_commit"]
    if cfg["adapter"] == "LayaTextDecider":
        kwargs["party_id"] = cfg["party_id"]
        kwargs["checkpoint_subdir"] = cfg.get("checkpoint_subdir", "")
    if SMOKE_MODE:
        kwargs["inference_fn"] = build_smoke_inference_fn(cfg["party_id"])
    elif cfg["adapter"] == "QwenVLDecider":
        kwargs["inference_fn"] = build_qwen_vl_inference_fn(cfg["hf_repo_id"], cfg["hf_revision"], use_acceleration=RUN_ACCELERATED)
    elif cfg["adapter"] == "LayaTextDecider":
        kwargs["inference_fn"] = build_laya_native_inference_fn(
            cfg["hf_repo_id"], cfg["hf_revision"],
            custom_code_audited=cfg["custom_code_audited"], audited_commit=cfg["audited_commit"],
            use_acceleration=RUN_ACCELERATED, checkpoint_subdir=cfg.get("checkpoint_subdir", ""),
        )
    elif cfg["adapter"] == "DiffusionVLMDecider":
        kwargs["inference_fn"] = build_diffusion_vlm_inference_fn(
            cfg["hf_repo_id"], cfg["hf_revision"],
            custom_code_audited=cfg["custom_code_audited"], audited_commit=cfg["audited_commit"],
            use_acceleration=RUN_ACCELERATED,
        )
    decider = ADAPTER_MAP[cfg["adapter"]](**kwargs)
    decider.validate_custom_code_gate()
    return decider


RAW_COMPARISONS = []
SKIPPED_MODELS = []
RESUMED_GROUPS = []
COMPLETED_GROUPS = []
for cfg in MODEL_CONFIGS:
    if RUN_PARTIES is not None and cfg["party_id"] not in RUN_PARTIES:
        continue
    if cfg["adapter"] in ("LayaTextDecider", "DiffusionVLMDecider") and (not cfg.get("custom_code_audited") or not cfg.get("audited_commit")):
        reason = (
            f"CUSTOM_CODE_NOT_AUDITED: {cfg['party_id']} ({cfg['hf_repo_id']}) requires "
            "custom_code_audited=True and pinned audited_commit before loading."
        )
        SKIPPED_MODELS.append({"party_id": cfg["party_id"], "model_id": cfg["hf_repo_id"], "adapter": cfg["adapter"],
                               "status": "SKIPPED_UNAUDITED", "reason": reason})
        print(f"[SKIPPED_UNAUDITED] {cfg['party_id']} ({cfg['hf_repo_id']}): {reason}")
        continue

    decider = None
    hf_prov = None
    for mode in cfg["modes"]:
        if mode not in INPUT_MODES:
            continue
        label = f"{cfg['party_id']}/{mode}"
        saved = load_group_checkpoint(OUTPUT_DIR, cfg["party_id"], mode, FINGERPRINT)
        if saved is not None:
            print(f"[RESUME] {label}: checkpoint found, skipping.")
            RAW_COMPARISONS.append(saved)
            RESUMED_GROUPS.append(label)
            COMPLETED_GROUPS.append(label)
            continue
        if decider is None:  # model is loaded only when some mode still has to run
            try:
                decider = build_decider(cfg)
            except CustomCodeAuditError as exc:
                SKIPPED_MODELS.append({"party_id": cfg["party_id"], "model_id": cfg["hf_repo_id"], "adapter": cfg["adapter"],
                                       "status": "SKIPPED_UNAUDITED", "reason": str(exc)})
                print(f"[SKIPPED_UNAUDITED] {cfg['party_id']} ({cfg['hf_repo_id']}): {exc}")
                break
            if SMOKE_MODE:
                hf_prov = {"hf_repo_id": cfg["hf_repo_id"], "requested_revision": cfg["hf_revision"], "resolved_commit_sha": "SMOKE_MODE",
                           "weight_files_sha256": {}, "weights_sha256": "UNKNOWN", "weights_unknown_reason": "SMOKE_MODE: no weights loaded"}
            else:
                # Resolved lazily, after warmup has downloaded/loaded the weights (the first model of a repo has an empty cache before that).
                _cfg = cfg
                hf_prov = lambda _c=_cfg: resolve_hf_provenance(_c["hf_repo_id"], _c["audited_commit"] or _c["hf_revision"])
        print(f"[RUN] {label} ...")
        res = run_acceleration_comparison(
            decider, REQUESTS,
            dataset_scope=DATASET_SCOPE, input_mode=mode,
            accelerated_label=ACCELERATED_DTYPE, unaccelerated_label="cpu",
            unaccelerated_sample_cap=CPU_SAMPLE_CAP, hf_provenance=hf_prov,
            run_unaccelerated=RUN_UNACCELERATED,
        )
        save_group_checkpoint(OUTPUT_DIR, cfg["party_id"], mode, res, FINGERPRINT)  # survives a runtime crash in a later group
        RAW_COMPARISONS.append(res)
        COMPLETED_GROUPS.append(label)
        print(f"[CHECKPOINT] {label} saved.")
print(f"Completed {len(RAW_COMPARISONS)} decider comparison runs ({len(RESUMED_GROUPS)} resumed); skipped {len(SKIPPED_MODELS)} unaudited models (SMOKE_MODE={SMOKE_MODE}).")

In [ ]:
# === CELL 7: SCORING CELL ONLY (ONLY CELL ALLOWED TO READ D385 LABEL-ONLY ANSWER TABLE) ===
# SCORE_ON_COLAB=False (per-tick pack, #767): no answer table is uploaded; this cell does nothing and scoring runs on the Mac.
if not SCORE_ON_COLAB:
    SCORED_MODELS, E7_REPORT = [], None
    print("SCORE_ON_COLAB=False: skipping scoring (no answer table on Colab).")
else:
    import json
    from amidst.evaluation.e7_scoring_bridge import build_e7_analysis_report, score_e7_comparison

    answer_table_path = PACK_ROOT / "scoring_only" / "answer_table.json"
    if not answer_table_path.is_file():
        raise FileNotFoundError(
            f"MISSING_ANSWER_TABLE: {answer_table_path} is required for Cell 7 scoring (D385). "
            "Refusing to silently fall back to an empty answer table."
        )
    ANSWER_TABLE = json.loads(answer_table_path.read_text(encoding="utf-8"))

    frozen_e5_path = PACK_ROOT / "baselines" / "e5_report_frozen.json"
    FROZEN_E5 = json.loads(frozen_e5_path.read_text(encoding="utf-8")) if frozen_e5_path.is_file() else None

    SCORED_MODELS = [score_e7_comparison(c, ANSWER_TABLE) for c in RAW_COMPARISONS]
    E7_REPORT = build_e7_analysis_report(
        hardware=HARDWARE_PROVENANCE,
        manifest_verification=MANIFEST_VERIFICATION,
        scored_models=SCORED_MODELS,
        dataset_scope=DATASET_SCOPE,
        frozen_e5_report=FROZEN_E5,
        skipped_models=SKIPPED_MODELS if "SKIPPED_MODELS" in globals() else [],
    )
    print("Scored", len(SCORED_MODELS), "model runs. Identity status:", E7_REPORT["identity_status"])


In [ ]:
# === CELL 8: WRITE e7_analysis_<date>.json, .md, AND PNG CHARTS (or the raw comparisons when scoring is off Colab) ===
import json as _json
from amidst.evaluation.e7_scoring_bridge import build_hardware_used, write_e7_outputs

if E7_REPORT is not None:
    OUT_PATHS = write_e7_outputs(E7_REPORT, OUTPUT_DIR)
else:
    _raw = OUTPUT_DIR / "e7_raw_comparisons.json"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    _out = [{**c, "hardware_used": build_hardware_used(c, HARDWARE_PROVENANCE)} for c in RAW_COMPARISONS]
    _raw.write_text(_json.dumps(_out, ensure_ascii=False) + "\n", encoding="utf-8")
    OUT_PATHS = {"raw_comparisons": _raw}
for kind, path in OUT_PATHS.items():
    print(f"Wrote {kind}: {path}")


In [ ]:
# === CELL 9: PACKAGE RUN (outputs + checkpoints + RUN_NOTES.json -> one folder + one zip on Drive) ===
import datetime as _dt
from amidst.evaluation.e7_colab_runtime import build_run_notes, package_run

RUN_NOTES = build_run_notes(
    notebook_version=NOTEBOOK_VERSION_DATE,
    git_commit=HARDWARE_PROVENANCE.get("git_commit", "UNKNOWN"),
    python_version=HARDWARE_PROVENANCE.get("python_version", "UNKNOWN"),
    pinned_deps=PINNED_DEPS,
    versions=RUNTIME_VERSIONS,
    pack_zip_sha256=PACK_ZIP_SHA256,
    manifest_verification=MANIFEST_VERIFICATION,
    smoke_mode=SMOKE_MODE,
    completed_groups=COMPLETED_GROUPS,
    resumed_groups=RESUMED_GROUPS,
    skipped_models=SKIPPED_MODELS,
    hf_provenance={f"{c['party_id']}/{c['input_mode']}": c.get("hf_provenance") for c in RAW_COMPARISONS},
    deviations=RUN_DEVIATIONS,
)
_tag = _dt.datetime.now(_dt.timezone.utc).strftime("%Y%m%d")
PACKAGED = package_run(OUTPUT_DIR, f"e7-run-{_tag}", RUN_NOTES, extra_files=list(OUT_PATHS.values()))
print("Packaged run:", PACKAGED)
